[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/mlflow-certified/notebooks/day-07-model-registry.ipynb#scrollTo=a1b2c3d4)

---
# Day 7 · The MLflow Model Registry — Staging, Production, Archiving
**certified-journeys / mlflow-certified** · Practice · Model Lifecycle Management

> **Goal for today:** Register a trained model in the MLflow Model Registry, transition it through Staging → Production, add aliases and descriptions, and load it back by stage for inference.


In [ ]:
%pip install -q mlflow scikit-learn pandas numpy


## Step 1 · What is the Model Registry?

The **MLflow Model Registry** is a centralized store that adds a lifecycle layer on top of MLflow runs. While a *run* captures one experiment attempt, the Registry gives you:

| Concept | Description |
|---|---|
| **Registered Model** | A named entity (e.g. `iris-classifier`) grouping all versions |
| **Model Version** | Each registration creates a version (v1, v2, …) linked to a run artifact |
| **Stage** | `None` → `Staging` → `Production` → `Archived` (classic API) |
| **Alias** | Mutable pointer like `champion` (MLflow 2.x — preferred over stages) |
| **Tags / Description** | Free-text metadata on model or version |

The Registry is separate from experiment tracking — you *register* a model from a run's artifacts.


In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Use a local file-based tracking server — works in Colab without a running MLflow server
mlflow.set_tracking_uri("sqlite:///mlflow_registry_demo.db")
mlflow.set_experiment("day-07-model-registry")

# Load and split data
iris = load_iris(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

print("Data loaded. Training set size:", X_train.shape[0])


**What just happened?**
- We pointed MLflow at a local **SQLite backend** — this is what enables the Model Registry (it needs a database, not just file storage).
- **`set_tracking_uri` with sqlite://…** is the minimal single-node setup that supports the full Registry API.
- In production you'd use a dedicated MLflow server (`mlflow server --backend-store-uri postgresql://…`).


## Step 2 · Train a Model and Log It to a Run

Before you can register, you need a *run* with a logged model artifact. The `mlflow.sklearn.log_model` call saves the model under the run's artifact path and also captures a **model signature** (input/output schema).

| Parameter | Purpose |
|---|---|
| `sk_model` | The fitted scikit-learn object |
| `artifact_path` | Sub-directory name inside the run's artifact root |
| `registered_model_name` | If set, auto-registers after logging — shortcut |
| `input_example` | Sample input — used to infer signature automatically |


In [ ]:
MODEL_NAME = "iris-classifier"

with mlflow.start_run() as run:
    # Train v1 — low n_estimators
    clf_v1 = RandomForestClassifier(n_estimators=10, random_state=42)
    clf_v1.fit(X_train, y_train)
    acc_v1 = accuracy_score(y_test, clf_v1.predict(X_test))

    mlflow.log_params({"n_estimators": 10, "random_state": 42})
    mlflow.log_metric("accuracy", acc_v1)

    # Log model — does NOT register yet (we'll do that explicitly below)
    mlflow.sklearn.log_model(
        sk_model=clf_v1,
        artifact_path="model",
        input_example=X_train.head(3),  # auto-infers signature
    )
    run_id_v1 = run.info.run_id

print(f"Run v1 complete. run_id={run_id_v1}, accuracy={acc_v1:.4f}")


**What just happened?**
- **`input_example`** causes MLflow to auto-infer an `mlflow.types.schema.Schema` — the signature is stored alongside the model.
- We captured `run_id_v1` — we'll need it to construct the `runs:/` URI for registration.
- The model artifact lives at `runs:/<run_id>/model` but is **not yet in the Registry**.


## Step 3 · Register a Model from a Run URI

Registration is a two-step concept:
1. **`mlflow.register_model(model_uri, name)`** — creates the registered model (if new) and adds a version.
2. A version starts in stage `None` and waits for a human (or CI) to promote it.

```
model_uri format:  runs:/<run_id>/<artifact_path>
```

Alternatively, setting `registered_model_name` in `log_model` does both steps in one call — but the explicit two-step gives you more control (e.g. only register if tests pass).


In [ ]:
client = MlflowClient()

# Build the model URI pointing to the run artifact
model_uri_v1 = f"runs:/{run_id_v1}/model"

# Register — this creates 'iris-classifier' version 1
mv1 = mlflow.register_model(model_uri=model_uri_v1, name=MODEL_NAME)

print(f"Registered: name={mv1.name}, version={mv1.version}, stage={mv1.current_stage}")

# List all versions to confirm
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
for v in versions:
    print(f"  version={v.version}, stage={v.current_stage}, run_id={v.run_id[:8]}…")


**What just happened?**
- **`mlflow.register_model`** created a `RegisteredModel` named `iris-classifier` and linked version 1 to our run.
- The version starts in stage **`None`** — it has not been approved for any environment yet.
- **`search_model_versions`** is the programmatic equivalent of the Registry UI's version list.


## Step 4 · Transition a Version to Staging

Stage transitions represent deployment readiness gates:

| Stage | Meaning |
|---|---|
| `None` | Just registered, no approval |
| `Staging` | Ready for integration tests / QA |
| `Production` | Serving live traffic |
| `Archived` | Retired, kept for audit |

> **MLflow 2.x note:** Stage-based promotion is being deprecated in favor of **aliases** (covered in Step 6). For new projects, prefer aliases. Legacy stage API shown here for awareness.


In [ ]:
# Transition version 1 → Staging
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=mv1.version,
    stage="Staging",
    archive_existing_versions=False,  # don't auto-archive other Staging versions
)

# Fetch fresh metadata to confirm the stage
mv1_updated = client.get_model_version(name=MODEL_NAME, version=mv1.version)
print(f"Version {mv1_updated.version} is now in stage: {mv1_updated.current_stage}")


**What just happened?**
- **`transition_model_version_stage`** is a `MlflowClient` method — it's the programmatic equivalent of clicking "Transition to Staging" in the UI.
- **`archive_existing_versions=True`** is useful when you want only one Staging/Production version at a time (it auto-archives displaced versions).
- Always re-fetch the version object after a transition — the original `mv1` reference holds stale stage data.


## Step 5 · Train a Better Model, Register v2, Promote to Production

In a real workflow, you'd run evaluations on the Staging version before promoting to Production. Here we simulate that by training a better v2 model, verifying its accuracy is higher, and promoting it.


In [ ]:
# Train v2 — more estimators, should be more accurate
with mlflow.start_run() as run:
    clf_v2 = RandomForestClassifier(n_estimators=100, random_state=42)
    clf_v2.fit(X_train, y_train)
    acc_v2 = accuracy_score(y_test, clf_v2.predict(X_test))

    mlflow.log_params({"n_estimators": 100, "random_state": 42})
    mlflow.log_metric("accuracy", acc_v2)

    mlflow.sklearn.log_model(
        sk_model=clf_v2,
        artifact_path="model",
        input_example=X_train.head(3),
    )
    run_id_v2 = run.info.run_id

print(f"v1 accuracy: {acc_v1:.4f} | v2 accuracy: {acc_v2:.4f}")

# Register v2
mv2 = mlflow.register_model(model_uri=f"runs:/{run_id_v2}/model", name=MODEL_NAME)
print(f"Registered v2: version={mv2.version}, stage={mv2.current_stage}")

# Gate: only promote if v2 is better than v1
if acc_v2 >= acc_v1:
    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=mv2.version,
        stage="Production",
        archive_existing_versions=True,  # archive any existing Production versions
    )
    print(f"v2 promoted to Production (acc={acc_v2:.4f} >= {acc_v1:.4f})")
else:
    print("v2 did not improve — keeping v1 in Production")


**What just happened?**
- **`archive_existing_versions=True`** ensures there is only one Production version at a time — the old one is automatically moved to `Archived`.
- The conditional gate (`if acc_v2 >= acc_v1`) models a real CI quality check — never promote a regression automatically.
- You can chain multiple `transition_model_version_stage` calls or build a full approval workflow on top of the Registry API.


## Step 6 · Add Description and Alias to the Production Version

**Aliases** (MLflow 2.x) are mutable named pointers to a specific version, independent of stages:

| Alias | Typical use |
|---|---|
| `champion` | Current best model in production |
| `challenger` | Candidate being shadow-tested |
| `latest` | Most recently trained version |

Aliases are more flexible than stages because multiple aliases can point to different versions simultaneously, and you can define your own lifecycle semantics.


In [ ]:
# Add a human-readable description to v2
client.update_model_version(
    name=MODEL_NAME,
    version=mv2.version,
    description=(
        f"RandomForest with n_estimators=100. "
        f"Accuracy={acc_v2:.4f} on held-out Iris test set. "
        "Promoted after beating v1 in accuracy gate."
    ),
)

# Set the 'champion' alias (MLflow 2.x API)
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="champion",
    version=mv2.version,
)

# Set 'challenger' alias on v1 so shadow testing can compare
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="challenger",
    version=mv1.version,
)

print("Aliases set. Resolving them back to version numbers:")
champion_ver = client.get_model_version_by_alias(MODEL_NAME, "champion")
challenger_ver = client.get_model_version_by_alias(MODEL_NAME, "challenger")
print(f"  champion  → version {champion_ver.version}")
print(f"  challenger → version {challenger_ver.version}")


**What just happened?**
- **`update_model_version`** also accepts `description` — always add context so team members understand why a version was promoted.
- **`set_registered_model_alias`** creates or moves the alias atomically — reassigning `champion` in CI requires no downtime.
- **`get_model_version_by_alias`** resolves the alias name back to a concrete version — useful in deployment scripts.


## Step 7 · Load a Model by Stage and by Alias

The Registry URI format for loading:

```
models:/<model-name>/<stage>          # by stage  (deprecated path)
models:/<model-name>@<alias>          # by alias  (MLflow 2.x preferred)
models:/<model-name>/<version-number> # by exact version
```

Loading by alias decouples your serving code from version numbers — you only need to update the alias in the Registry when you promote a new model.


In [ ]:
# Load the Production model by stage (classic approach)
prod_model_stage = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/Production")

# Load the champion model by alias (MLflow 2.x preferred)
prod_model_alias = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")

# Run inference with both — results should be identical
preds_stage = prod_model_stage.predict(X_test)
preds_alias = prod_model_alias.predict(X_test)

print("Predictions from stage URI match alias URI:", (preds_stage == preds_alias).all())
print("Accuracy (stage load):", accuracy_score(y_test, preds_stage))
print("First 5 predictions:", preds_stage[:5].tolist())
print("Actual labels:       ", y_test[:5].tolist())


**What just happened?**
- **`models:/<name>/Production`** and **`models:/<name>@champion`** both resolve to version 2 — confirming alias and stage point to the same artifact.
- Loading by alias (`@champion`) is the **forward-compatible** approach: when stages are deprecated in a future MLflow release, alias-based loading will continue to work without code changes.
- The loaded model is a standard scikit-learn object — `.predict()`, `.predict_proba()`, etc. all work normally.


## Step 8 · Archive Old Versions and Inspect Registry State

Archiving is the final lifecycle stage — it retires a version without deleting it (preserving audit history). You should archive Staging versions that were overtaken, or any version no longer in active use.


In [ ]:
# Archive v1 (it's now the challenger but no longer needed in Staging)
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=mv1.version,
    stage="Archived",
)

# Print a full Registry summary
print(f"\n=== Registry summary for '{MODEL_NAME}' ===")
all_versions = client.search_model_versions(f"name='{MODEL_NAME}'")
for v in sorted(all_versions, key=lambda x: int(x.version)):
    print(f"  v{v.version} | stage={v.current_stage:12s} | run={v.run_id[:8]}… | {v.description[:50] if v.description else '(no description)'}")

# Show registered model-level metadata
rm = client.get_registered_model(MODEL_NAME)
print(f"\nRegistered model: {rm.name}")
print(f"Latest versions: {[(v.version, v.current_stage) for v in rm.latest_versions]}")


**What just happened?**
- **`Archived`** stage preserves the version in the Registry without making it loadable via `models:/<name>/Production` — the artifact is still accessible by exact version number.
- **`get_registered_model`** returns model-level metadata including `latest_versions` — a list of the most recent version per stage.
- In production, archiving should be automated by CI: when a new version reaches Production, archive the previously-Production version.


In [ ]:
# Challenge: Build a promotion pipeline
#
# Task: Write a function `safe_promote(client, model_name, candidate_version, baseline_accuracy)`
# that:
#   1. Fetches the run linked to `candidate_version` and reads its 'accuracy' metric
#   2. If candidate accuracy > baseline_accuracy, transitions to Production
#      (archive_existing_versions=True) AND sets alias 'champion'
#   3. If not better, transitions to Archived and prints a reason
#   4. Returns True if promoted, False otherwise
#
# Hints:
#   - client.get_model_version(name, version) → .run_id
#   - client.get_run(run_id).data.metrics['accuracy']
#   - client.transition_model_version_stage(...)
#   - client.set_registered_model_alias(...)

def safe_promote(client, model_name, candidate_version, baseline_accuracy):
    # Your solution here
    pass

# Test it (uncomment after implementing):
# result = safe_promote(client, MODEL_NAME, mv2.version, acc_v1)
# print("Promoted:", result)


---
## Day 7 key concepts recap

| Concept | What to remember |
|---|---|
| `mlflow.register_model(uri, name)` | Registers an artifact from a run; creates the model if it doesn't exist |
| `MlflowClient.transition_model_version_stage` | Moves a version through None → Staging → Production → Archived |
| `archive_existing_versions=True` | Ensures only one Production version exists at a time |
| `set_registered_model_alias` | MLflow 2.x: mutable named pointer (champion, challenger) preferred over stages |
| `models:/<name>@<alias>` | Load-by-alias URI — decouples serving code from version numbers |
| `update_model_version` | Add description and tags to document why a version was promoted |

> **Tip:** Use model aliases like `champion` and `challenger` (MLflow 2.x) instead of stage names — aliases are more flexible and stage deprecation is planned.

---
## What's next
**Day 8** → MLflow Projects — packaging experiments as reproducible, shareable, parameterized pipelines runnable with a single `mlflow run` command.

Mark Day 7 complete in your [tracker](../index.html).
